## Import libraries

In [ ]:
import llm_funcs.funcs as llm_helpers
import llm_funcs.langchain_funcs as langchain_helpers

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.runnables import RunnablePassthrough
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain.chains import LLMChain, RetrievalQA
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_ibm import WatsonxEmbeddings
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames

import os

## Try a basic prompt

In [2]:
# Modify params
params = {
    "max_new_tokens": 24,
    "min_new_tokens": 10,
    "temperature": 0.5,
    "top_p": 0.2,
    "top_k": 1
}

# Define prompt and get response from LLM
prompt = "The wind is"
llm = langchain_helpers.llm_model(prompt, "mistralai/mistral-small-3-1-24b-instruct-2503", params)
response = llm.invoke(prompt)

# Print results
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: The wind is

response :  blowing at 20 mph. The wind chill is 10 degrees Fahrenheit. What is the temperature?





## Deploy prompt templates and chains using LangChain

#### Start by a simple case

In [3]:
# Get model
parameters = {'max_new_tokens': 26,
              'temperature': 0.5}
llm = langchain_helpers.llm_model(prompt, "mistralai/mistral-small-3-1-24b-instruct-2503", parameters)

# Define template
template = """Tell me a {adjective} joke about {content}."""
prompt = PromptTemplate.from_template(template)

# Format according to current case
prompt.format(adjective="funny", content="chickens")

# Wrap formatted template into LLMChain
llm_chain = LLMChain(prompt=prompt, llm=llm)

# Get response
response = llm_chain.invoke(input = {"adjective": "scary", "content": "cars"})

# Print response
print(response['text'])


 What do you call a car with no doors? A taxi.

What do you call a car with no engine? A Toyota.




#### One-short learning case for LLM prompting using LangChain

In [4]:
# Define model
parameters = {'max_new_tokens': 10,
              'temperature': 0.5}
llm = langchain_helpers.llm_model(prompt, "mistralai/mistral-small-3-1-24b-instruct-2503", parameters)

# Define examples given to the LLM for one-short learning
example_text = """Tomorrow we are going to have dinner to the new restaurant in town."""
example_category = """"Food and Dining."""

# Define the prompt
text = """The concert last night was really fun in spite of the bad weather."""
categories = """Entertainment, Food and Dining, Technology, Literature, Music."""

# Define template
template = """Example: {example_text}
              Category: {example_category}
            
              Now, classify the following sentence: {text} into one of the following categories: {categories}.
              Category: """
prompt = PromptTemplate.from_template(template)

output_key = "category"

# Wrap
llm_chain = LLMChain(prompt=prompt, llm=llm, output_key=output_key)

# Get answer
response = llm_chain.invoke(input = {"text":text ,"categories": categories, "example_text": example_text, "example_category": example_category})

# Print answer
print(response["category"])

 Music

1. **Sentence**: We are


## Additional functionalities of LangChain

In [5]:
### LangChain enables prompt engineering by using few-shot prompts
## To generate the prompt
# Examples given for few-shot trials
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input": "windy", "output": "calm"},
]

# Template for each example
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)

# Selector to avoid excessive token usage
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=25,  # The maximum length that the formatted examples should be.
)

# Generate prompt
dynamic_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Give the antonym of every input",
    suffix="Input: {adjective}\nOutput:",
    input_variables=["adjective"],
)

## Now we can invoke the LLM on the prompt
llm_chain = LLMChain(prompt=dynamic_prompt, llm=llm)
response = llm_chain.invoke(input = {'adjective': 'big'})

## And we can print the result
print(f"{dynamic_prompt.format(adjective='big')}{response["text"]}")

Give the antonym of every input

Input: happy
Output: sad

Input: tall
Output: short

Input: energetic
Output: lethargic

Input: sunny
Output: gloomy

Input: windy
Output: calm

Input: big
Output: small

Input: fast
Output: slow




In [ ]:
# LangChain also supports for output in specific formats
output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="Answer the user query. {format_instructions}\nList five {subject}.",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

# Alternative notation to LangChain chains
chain = prompt | llm | output_parser

chain.invoke({"subject": "ice cream flavors"})

['Vanilla', 'Chocolate', 'Strawberry', 'Mint Chocolate']

## LangChain enables high-level usage of LLMs to process text

In [20]:
# Use LangChain to read a pdf file into a document with content and metadata
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# Split the text into smaller chunks adapted to LLM's context window
text_splitter = CharacterTextSplitter(chunk_size=400, chunk_overlap=20, separator="\n")  # define chunk_size which is length of characters, and also separator.
chunks = text_splitter.split_documents(document)

# Text chunks can be stored and embedded for later retrieval
# Define embedding parameters
embed_params = {
    EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3,
    EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
}

# Get embedding
watsonx_embedding = WatsonxEmbeddings(
    model_id="ibm/slate-125m-english-rtrvr-v2",
    url=os.getenv("WATSONX_URL"),
    project_id=os.getenv("WATSONX_PROJECT_ID"),
    params=embed_params,
)

# Embed chunks
docsearch = Chroma.from_documents(chunks, watsonx_embedding)

# Define model
parameters = {'max_new_tokens': 50,
              'temperature': 0.5}
llm = langchain_helpers.llm_model(prompt, "mistralai/mistral-small-3-1-24b-instruct-2503", parameters)


# Create QA chatbot
qa = RetrievalQA.from_chain_type(llm=llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)

# Query and get answer
query = "what is this paper discussing?"
response = qa.invoke(query)

# Print query and results
print(response['query'])
print(response['result'])

what is this paper discussing?
 This paper is discussing the application of recent advancements in pretrained contextualized large language models to introduce MindGuide, an innovative chatbot designed to function as a mental health assistant for individuals in need of guidance and support.


## Sequential chains are also widely used in LangChain

In [22]:
# Chain 1
location_template = """Your job is to come up with a classic dish from the area that the users suggests.
                {location}
                
                YOUR RESPONSE:
"""
location_prompt_template = PromptTemplate(template=location_template, input_variables=['location'])

# Chain 2
meal_template = """Given a meal {meal}, give a short and simple recipe on how to make that dish at home.

                YOUR RESPONSE:
"""
meal_prompt_template = PromptTemplate(template=meal_template, input_variables=['meal'])


# Chain 3
recipe_template = """Given the recipe {recipe}, estimate how much time I need to cook it.

                YOUR RESPONSE:
"""
recipe_prompt_template = PromptTemplate(template=recipe_template, input_variables=['recipe'])

# overall chain
location_chain = location_prompt_template | llm
dish_chain = meal_prompt_template | llm
recipe_chain = recipe_prompt_template | llm

overall_chain = (
    RunnablePassthrough.assign(meal=location_chain)
    .assign(recipe=dish_chain)
    .assign(time=recipe_chain)
)

response = overall_chain.invoke({"location": "Barcelona"})
print(response)

{'location': 'Barcelona', 'meal': "                A classic dish from Barcelona is **Pa amb Tomàquet**. This traditional Catalan dish consists of toasted bread rubbed with garlic and ripe tomato, drizzled with olive oil, and sprinkled with salt. It's a simple yet flavorful", 'recipe': 'To make **Pa amb Tomàquet** at home, follow these simple steps:\n\n**Ingredients:**\n- 1 baguette or rustic bread\n- 1-2 ripe tomatoes\n- 1-2 cloves of garlic\n-', 'time': "To make **Pa amb Tomàquet** at home, you will need approximately 10-15 minutes of active preparation time. Here's a breakdown of the time required:\n\n1. **Preparation (5 minutes):**\n   - G"}
